In [1]:
import pandas as pd
import geopandas as gpd
import json

#show all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

Important notes about the columns in the Occupation Scores Study (`occ_scores`):
- Columns prefixed with `dv_rating_` are how GPT-4 rated the tasks/occupations
- Columns prefixed with `human_rating_` are how the human annotators rated the tasks/occupations
- Columns suffixed with `_alpha` correspond to the share of that occupation's tasks that would directly be made easier by AI. 
- Columns suffixed with `_beta` correspond to the share of that occupation's tasks that would be either directly made easier or would be made easier with additional tools built using AI. A half weight was given to tasks that would be made easier but only with additional AI tools.
- Columns suffixed with `_gamma` is similar to `_beta` but it gives full weight to those tasks made easier only with additional AI tools in the mix.
- According to the study, "made easier" mean that using LLMs via ChatGPT or the OpenAI playground would decrease the time required to complete the task by at least half (50%).

NOTE TO REPORTERS AND EDITORS: The initial analysis used the `human_rating_beta` because we thought it was the fairest approach to understanding AI's impact on jobs. If you prefer one of the other ratings for editorial reasons, you are free to use it. I do recommend that you read the full study before you make that decision.


Allie add the wider occupation name to all of this please! Maybe some aggregate analysis on how the wider occupations are fairing faring... flaring.

In [ ]:
# load data
## study data (has been downloaded to the raw data folder as well)
study_rating = 'human_rating_beta'
outfile = f'../data/processed/occ_study_scores_{study_rating}.csv'

occ_scores = pd.read_csv('https://raw.githubusercontent.com/openai/GPTs-are-GPTs/refs/heads/main/data/occ_level.csv')

#adding in the crosswalk to the O*NET SOC codes which apparently we DO need
xwalk = pd.read_excel('../data/raw/nem-onet-to-soc-crosswalk.xlsx', sheet_name='ONET to SOC crosswalk', skiprows=4)
xwalk = xwalk[['O*NET-SOC Code','O*NET-SOC Title','NEM Code','National Employment Matrix Occupational Title']]
xwalk = xwalk.rename(columns={'National Employment Matrix Occupational Title': 'NEM Title'})
print('Occ scores pre merge:', len(occ_scores))
occ_scores = occ_scores.merge(xwalk, how='left', on='O*NET-SOC Code')
print('Occ scores post merge w/ xwalk:', len(occ_scores))


occ_scores = occ_scores[['O*NET-SOC Code', 'Title','NEM Code','NEM Title', study_rating]]
print('Total occupation scores:', len(occ_scores))
occ_scores['occupation_code'] = occ_scores['O*NET-SOC Code'].str.replace('-', '')
occ_scores['occupation_code'] = occ_scores['occupation_code'].apply(lambda x: x if len(x) == 6 else x[:6])

occ_scores['nem_code_merge'] = occ_scores['NEM Code'].str.replace('-', '')

print('Occupation scores sans detailed:', len(occ_scores))


occ_scores = occ_scores.rename(columns={study_rating: 'study_rating', 
                                        'O*NET-SOC Code': 'soc_code',
                                        'Title': 'occupation_name'})

## BLS employment estimates from the data download step
oews = pd.read_csv('../data/processed/bls_oews_current_employment.csv', dtype={'occupation_code': str, 'area_code': str, 'datatype_code': str})

Occ scores pre merge: 923
Occ scores post merge w/ xwalk: 923
Total occupation scores: 923
Occupation scores sans detailed: 923


In [ ]:
#check it out if it pleases m'lord
oews.sample(5)

In [ ]:
oews.loc[oews['occupation_name'].str.contains('home health', case=False)].head()

In [ ]:
#see which columns have NAs
print(oews.isna().sum())

In [ ]:
oews.loc[oews['employment'].isna()].sample(2)

In [ ]:
oews.loc[(oews['employment'].isna())].groupby('footnote_codes',dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)

Footnote code 8 means that the estimate not released.

In [ ]:
#let's look at a subset of these occupations for our different markets and compare against the BLS website
#to make sure there really isn't data
markets = ['San Francisco-Oakland-Fremont, CA','Albany-Schenectady-Troy, NY','Houston-Pasadena-The Woodlands, TX']
oews.loc[(oews['area_name'].isin(markets))&(oews['employment'].isna())].sample(5)

In [ ]:
oews.loc[(oews['soc_code'].isna())]['occupation_name'].unique()

**NOTE ABOUT NAs in the `employment` column:** I've checked a handful of the records with NA in the value column against what BLS provides on [their website](https://data.bls.gov/oes/#/home) via the one-screen database search and seems like they are all `(8) Estimate not released.`

**NOTE ABOUT NAs in the `occupation_name` column:** These are all total employment values. Total employment was not scored by the study.

In [ ]:
oews.loc[oews['occupation_name'].str.contains('home health', case=False)].head()

In [ ]:
occ_scores.loc[occ_scores['occupation_name'].str.contains('home health', case=False)].head()

In [ ]:
# Clean up and merge

#first we wanna pull out detailed occupations from the study because we won't have employment
#data for those. But we do wanna keep them for the nested table later
detailed_scores = occ_scores[~occ_scores['soc_code'].str.endswith('.00')]
general_scores = occ_scores[occ_scores['soc_code'].str.endswith('.00')]

#make sure we don't have dups in our general_scores now
alert_message = "WARNING: Duplicates found in general scores table!"
assert general_scores['occupation_code'].nunique() == general_scores.shape[0], alert_message

#merge only the general scores so we're not getting false employment matches
oews_with_scores = oews.merge(general_scores, left_on='occupation_code', right_on='nem_code_merge', how='left', suffixes=('_oews', '_study'))

In [ ]:
print(len(oews), 'rows in the original table')
print(len(oews_with_scores), 'rows in the merged table')

In [ ]:
#what does sf home health aide look like?
oews_with_scores.loc[(oews_with_scores['occupation_name_study'].str.contains('home health', case=False))&(oews_with_scores['area_name'].str.contains('San Francisco', case=False))].head()

In [ ]:
#oews_with_scores.columns
keep_cols =  ['series_id', 'year', 'period', 'areatype_code',
              'state_code', 'area_code', 'area_name', 'occupation_code_oews',
              'occupation_name_oews', 'employment', 'study_rating',
              'footnote_codes', 'is_all_occupations', 'has_released_employment',
              'occupation_code_study','occupation_name_study', 
              'NEM Code', 'NEM Title','nem_code_merge']

oews_with_scores = oews_with_scores[keep_cols]

## Testing crosswalk merge
Ok so the crosswalk merge does create duplicates. I mean duh we knew it would.

My one idea was that after the join, we could groupby area_code and drop duplicates, keeping the occ code with the highest employment. I suppose that's a bit arbitrary though. 

But first, a big question for me right now is how prevelant is this? I'm proposing you do a brief investigation that looks at how often, for our metros and states, this happens and to which occupations.

In [3]:
occ_scores.columns

Index(['soc_code', 'occupation_name', 'NEM Code', 'NEM Title', 'study_rating',
       'occupation_code', 'nem_code_merge'],
      dtype='str')

In [39]:
def find_exact_matches(df, soc_col):
    """
    This function takes a dataframe and a column name as input and returns the number of exact matches in that column.
    An exact match is defined as a value that ends with '.00'.
    """
    nem_expanded = df['nem_code_merge'].str[:2]+ '-'+ df['nem_code_merge'].str[2:] + '.00'
    return ((df[soc_col] == nem_expanded)).sum()

def calculate_stats(df):
    """
    This function takes a dataframe as input and returns a dictionary with the minimum, maximum, and mean of the 'study_rating' column.
    """
    
    return pd.Series({
        'soc_cnt': df['soc_code'].nunique(),
        'socs': ';'.join(df['soc_plus_score'].unique()),
        'exact_soc_match': find_exact_matches(df, 'soc_code'),
        'study_rating_min': df['study_rating'].min(),
        'study_rating_max': df['study_rating'].max(),
        'study_rating_mean': df['study_rating'].mean()
    })

occ_scores['soc_plus_score'] = occ_scores['occupation_name'] + ' (' + occ_scores['study_rating'].astype(float).round(2).astype(str) + ')'
scores_by_nem = occ_scores.groupby(['NEM Code', 'NEM Title']).apply(calculate_stats).reset_index().sort_values('soc_cnt', ascending=False)

In [40]:
scores_by_nem.head(76)

,NEM Code,NEM Title,soc_cnt,socs,exact_soc_match,study_rating_min,study_rating_max,study_rating_mean
83,15-1299,"Computer occupations, all other",9,Web Administrators (0.67);Geographic Information Systems Technologists and Technicians (0.6);Document Management Specialists (0.68);Penetration Testers (0.59);Information Security Engineers (0.62);Digital Forensics Analysts (0.65);Blockchain Engineers (0.53);Computer Systems Engineers/Architects (0.57);Information Technology Project Managers (0.45),0,0.452381,0.677778,0.596964
111,17-2199,"Engineers, all other",8,"Energy Engineers, Except Wind and Solar (0.5);Mechatronics Engineers (0.3);Microsystems Engineers (0.53);Photonics Engineers (0.29);Robotics Engineers (0.48);Nanosystems Engineers (0.49);Wind Energy Engineers (0.53);Solar Energy Systems Engineers (0.46)",0,0.288889,0.526316,0.446173
41,13-1041,Compliance officers,7,Compliance Officers (0.61);Environmental Compliance Inspectors (0.49);Equal Opportunity Representatives and Officers (0.5);Government Property Inspectors and Investigators (0.35);Coroners (0.46);Regulatory Affairs Specialists (0.67);Customs Brokers (0.6),1,0.352941,0.669643,0.526537
324,29-1229,"Physicians, all other",6,Allergists and Immunologists (0.37);Hospitalists (0.36);Urologists (0.25);Physical Medicine and Rehabilitation Physicians (0.13);Preventive Medicine Physicians (0.37);Sports Medicine Physicians (0.43),0,0.133333,0.433962,0.317962
36,11-9199,"Managers, all other",6,Regulatory Affairs Managers (0.54);Compliance Managers (0.54);Loss Prevention Managers (0.37);Wind Energy Operations Managers (0.41);Wind Energy Development Managers (0.53);Brownfield Redevelopment Specialists and Site Managers (0.41),0,0.367925,0.540000,0.465491
12,11-3051,Industrial production managers,6,Industrial Production Managers (0.35);Quality Control Systems Managers (0.47);Geothermal Production Managers (0.38);Biofuels Production Managers (0.18);Biomass Power Plant Managers (0.28);Hydroelectric Production Managers (0.31),1,0.178571,0.471698,0.329604
331,29-2010,Clinical laboratory technologists and technicians,6,Medical and Clinical Laboratory Technologists (0.23);Cytogenetic Technologists (0.34);Cytotechnologists (0.16);Histotechnologists (0.06);Medical and Clinical Laboratory Technicians (0.1);Histology Technicians (0.0),0,0.000000,0.336207,0.147681
307,29-1141,Registered nurses,5,Registered Nurses (0.38);Acute Care Nurses (0.33);Advanced Practice Psychiatric Nurses (0.44);Critical Care Nurses (0.19);Clinical Nurse Specialists (0.33),1,0.189655,0.444444,0.334983
141,19-2041,"Environmental scientists and specialists, including health",4,"Environmental Scientists and Specialists, Including Health (0.73);Climate Change Policy Analysts (0.75);Environmental Restoration Planners (0.65);Industrial Ecologists (0.71)",1,0.652174,0.750000,0.710304
740,53-1047,"First-line supervisors of transportation and material moving workers, except aircraft cargo handling supervisors",4,"First-Line Supervisors of Helpers, Laborers, and Material Movers, Hand (0.36);Recycling Coordinators (0.47);First-Line Supervisors of Material-Moving Machine and Vehicle Operators (0.31);First-Line Supervisors of Passenger Attendants (0.32)",0,0.307692,0.465517,0.361139


In [41]:
print('Total unique NEM codes in the study:', len(scores_by_nem))
print('NEM with more than one SOC match:', len(scores_by_nem.loc[scores_by_nem['soc_cnt'] > 1]))
print('NEM with more than one SOC match but also an exact match:', len(scores_by_nem.loc[(scores_by_nem['soc_cnt'] > 1) & (scores_by_nem['exact_soc_match'] > 0)]))
print('NEM with more than one SOC match but NO exact match:', len(scores_by_nem.loc[(scores_by_nem['soc_cnt'] > 1) & (scores_by_nem['exact_soc_match'] == 0)]))

Total unique NEM codes in the study: 785
NEM with more than one SOC match: 75
NEM with more than one SOC match but also an exact match: 49
NEM with more than one SOC match but NO exact match: 26


In [42]:
#show me the NEMs with more than one SOC match and no exact matches
multi_match_no_exact = scores_by_nem.loc[(scores_by_nem['soc_cnt'] > 1) & (scores_by_nem['exact_soc_match'] == 0)]
multi_match_no_exact

,NEM Code,NEM Title,soc_cnt,socs,exact_soc_match,study_rating_min,study_rating_max,study_rating_mean
83,15-1299,"Computer occupations, all other",9,Web Administrators (0.67);Geographic Information Systems Technologists and Technicians (0.6);Document Management Specialists (0.68);Penetration Testers (0.59);Information Security Engineers (0.62);Digital Forensics Analysts (0.65);Blockchain Engineers (0.53);Computer Systems Engineers/Architects (0.57);Information Technology Project Managers (0.45),0,0.452381,0.677778,0.596964
111,17-2199,"Engineers, all other",8,"Energy Engineers, Except Wind and Solar (0.5);Mechatronics Engineers (0.3);Microsystems Engineers (0.53);Photonics Engineers (0.29);Robotics Engineers (0.48);Nanosystems Engineers (0.49);Wind Energy Engineers (0.53);Solar Energy Systems Engineers (0.46)",0,0.288889,0.526316,0.446173
324,29-1229,"Physicians, all other",6,Allergists and Immunologists (0.37);Hospitalists (0.36);Urologists (0.25);Physical Medicine and Rehabilitation Physicians (0.13);Preventive Medicine Physicians (0.37);Sports Medicine Physicians (0.43),0,0.133333,0.433962,0.317962
36,11-9199,"Managers, all other",6,Regulatory Affairs Managers (0.54);Compliance Managers (0.54);Loss Prevention Managers (0.37);Wind Energy Operations Managers (0.41);Wind Energy Development Managers (0.53);Brownfield Redevelopment Specialists and Site Managers (0.41),0,0.367925,0.540000,0.465491
331,29-2010,Clinical laboratory technologists and technicians,6,Medical and Clinical Laboratory Technologists (0.23);Cytogenetic Technologists (0.34);Cytotechnologists (0.16);Histotechnologists (0.06);Medical and Clinical Laboratory Technicians (0.1);Histology Technicians (0.0),0,0.000000,0.336207,0.147681
740,53-1047,"First-line supervisors of transportation and material moving workers, except aircraft cargo handling supervisors",4,"First-Line Supervisors of Helpers, Laborers, and Material Movers, Hand (0.36);Recycling Coordinators (0.47);First-Line Supervisors of Material-Moving Machine and Vehicle Operators (0.31);First-Line Supervisors of Passenger Attendants (0.32)",0,0.307692,0.465517,0.361139
54,13-1199,"Business operations specialists, all other",4,Business Continuity Planners (0.5);Sustainability Specialists (0.68);Online Merchants (0.56);Security Management Specialists (0.41),0,0.413043,0.678571,0.537990
131,19-1029,"Biological scientists, all other",4,Bioinformatics Scientists (0.55);Molecular and Cellular Biologists (0.44);Geneticists (0.48);Biologists (0.38),0,0.382353,0.545455,0.460894
351,29-2099,"Health technologists and technicians, all other",3,Neurodiagnostic Technologists (0.19);Ophthalmic Medical Technologists (0.07);Patient Representatives (0.57),0,0.066667,0.571429,0.275198
38,13-1020,Buyers and purchasing agents,3,"Buyers and Purchasing Agents, Farm Products (0.25);Wholesale and Retail Buyers, Except Farm Products (0.4);Purchasing Agents, Except Wholesale, Retail, and Farm Products (0.37)",0,0.250000,0.400000,0.339474


Ok so these 26 with more than one match AND no exact match are the ones we need to care about. I'm going to look at employment levels for these occupations in each of our market areas and see how big of an issue this is going to be.

In [43]:
market = 'San Francisco-Oakland-Fremont, CA'
min_cols = ['area_name', 'NEM Code', 'occupation_name', 'employment','footnote_codes',
            'soc_cnt','socs', 'exact_soc_match','study_rating_min', 'study_rating_max', 
            'study_rating_mean']
market_occs = oews.loc[oews['area_name'] == market]
#occs_affected = market_occs.loc[(market_occs['soc_code'].isin(multi_match_no_exact['NEM Code']))]
occs_affected = multi_match_no_exact.merge(market_occs, left_on='NEM Code', right_on='soc_code', how='inner', suffixes=('_study', '_oews'))
print('Occs affected in SF:', len(occs_affected))
display(occs_affected.sort_values('employment', ascending=False)[min_cols])

Occs affected in SF: 26


,area_name,NEM Code,occupation_name,employment,footnote_codes,soc_cnt,socs,exact_soc_match,study_rating_min,study_rating_max,study_rating_mean
10,"San Francisco-Oakland-Fremont, CA",31-1120,Home Health and Personal Care Aides,119120.0,NaN,2,Home Health Aides (0.04);Personal Care Aides (0.33),0,0.038462,0.333333,0.185897
6,"San Francisco-Oakland-Fremont, CA",13-1199,"Business Operations Specialists, All Other",23400.0,NaN,4,Business Continuity Planners (0.5);Sustainability Specialists (0.68);Online Merchants (0.56);Security Management Specialists (0.41),0,0.413043,0.678571,0.537990
3,"San Francisco-Oakland-Fremont, CA",11-9199,"Managers, All Other",21550.0,NaN,6,Regulatory Affairs Managers (0.54);Compliance Managers (0.54);Loss Prevention Managers (0.37);Wind Energy Operations Managers (0.41);Wind Energy Development Managers (0.53);Brownfield Redevelopment Specialists and Site Managers (0.41),0,0.367925,0.540000,0.465491
13,"San Francisco-Oakland-Fremont, CA",25-9045,"Teaching Assistants, Except Postsecondary",18710.0,NaN,2,"Teaching Assistants, Preschool, Elementary, Middle, and Secondary School, Except Special Education (0.28);Teaching Assistants, Special Education (0.25)",0,0.250000,0.283333,0.266667
0,"San Francisco-Oakland-Fremont, CA",15-1299,"Computer Occupations, All Other",16930.0,NaN,9,Web Administrators (0.67);Geographic Information Systems Technologists and Technicians (0.6);Document Management Specialists (0.68);Penetration Testers (0.59);Information Security Engineers (0.62);Digital Forensics Analysts (0.65);Blockchain Engineers (0.53);Computer Systems Engineers/Architects (0.57);Information Technology Project Managers (0.45),0,0.452381,0.677778,0.596964
16,"San Francisco-Oakland-Fremont, CA",21-1018,"Substance Abuse, Behavioral Disorder, and Mental Health Counselors",9520.0,NaN,2,Substance Abuse and Behavioral Disorder Counselors (0.29);Mental Health Counselors (0.26),0,0.264706,0.288889,0.276797
5,"San Francisco-Oakland-Fremont, CA",53-1047,"First-Line Supervisors of Transportation and Material Moving Workers, Except Aircraft Cargo Handling",7450.0,NaN,4,"First-Line Supervisors of Helpers, Laborers, and Material Movers, Hand (0.36);Recycling Coordinators (0.47);First-Line Supervisors of Material-Moving Machine and Vehicle Operators (0.31);First-Line Supervisors of Passenger Attendants (0.32)",0,0.307692,0.465517,0.361139
9,"San Francisco-Oakland-Fremont, CA",13-1020,Buyers and Purchasing Agents,6300.0,NaN,3,"Buyers and Purchasing Agents, Farm Products (0.25);Wholesale and Retail Buyers, Except Farm Products (0.4);Purchasing Agents, Except Wholesale, Retail, and Farm Products (0.37)",0,0.250000,0.400000,0.339474
4,"San Francisco-Oakland-Fremont, CA",29-2010,Clinical Laboratory Technologists and Technicians,5210.0,NaN,6,Medical and Clinical Laboratory Technologists (0.23);Cytogenetic Technologists (0.34);Cytotechnologists (0.16);Histotechnologists (0.06);Medical and Clinical Laboratory Technicians (0.1);Histology Technicians (0.0),0,0.000000,0.336207,0.147681
1,"San Francisco-Oakland-Fremont, CA",17-2199,"Engineers, All Other",4850.0,NaN,8,"Energy Engineers, Except Wind and Solar (0.5);Mechatronics Engineers (0.3);Microsystems Engineers (0.53);Photonics Engineers (0.29);Robotics Engineers (0.48);Nanosystems Engineers (0.49);Wind Energy Engineers (0.53);Solar Energy Systems Engineers (0.46)",0,0.288889,0.526316,0.446173


In [44]:
occs_affected.sort_values('employment', ascending=False)[min_cols].to_csv('../data/processed/sf/occs_multi_match_no_exact.csv', index=False)

In [ ]:
#testing the codes against each other
#soc_code_oews will always match occupation_code_oews because they are derived from each other. Same with soc_code_study and occupation_code_study
#also, the nem will always match the oews values because we merged on that
#so therefore if we have a match between occupation_code_oews and occupation_code_study, we can assume matches for the others too

oews_code_test = oews_with_scores.copy()
oews_code_test['study_v_oews'] = oews_code_test.apply(lambda x: 'match' if x['occupation_code_oews'] == x['occupation_code_study'] else 'mismatch', axis=1)
oews_code_test['study_v_nem'] = oews_code_test.apply(lambda x: 'match' if x['nem_code_merge'] == x['occupation_code_study'] else 'mismatch', axis=1)

In [ ]:
unique_pair_occ_codes = oews_code_test.groupby(['occupation_code_oews', 'occupation_code_study','study_v_oews'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)
unique_pair_occ_codes.groupby('study_v_oews').size()

In [ ]:
unique_pair_occ_codes = oews_code_test.groupby(['occupation_code_oews', 'occupation_code_study','study_v_nem'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)
unique_pair_occ_codes.groupby('study_v_nem').size()

In [ ]:
unique_pair_nem_codes = oews_code_test.groupby(['nem_code_merge', 'occupation_code_study','study_v_nem'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)
unique_pair_nem_codes.groupby('study_v_nem', dropna=False).size()

In [ ]:
print(len(oews_code_test))
print(len(oews_code_test.loc[(oews_code_test['study_v_oews'] == 'match') & (oews_code_test['study_v_nem'] == 'match')]))

In [ ]:
unique_pair_occ_codes.loc[unique_pair_occ_codes['occ_code_study_v_oews'] == 'mismatch'].sample(10)

In [ ]:
oews_with_scores.loc[oews_with_scores['occupation_code_oews'] == '131020'].sample(5)

## And categorization

In [ ]:
#categorize
def get_ai_exposure_category(score):
    if score <= .25:
        return 'Low'
    elif score <= .5:
        return 'Medium low'
    elif score <= .75:
        return 'Medium high'
    elif score > .75:
        return 'High'
    else:
        return 'NA'

oews_with_scores['ai_exposure_category'] = oews_with_scores['study_rating'].apply(get_ai_exposure_category)
oews_with_scores['employment_category'] = pd.qcut(oews_with_scores['value'], q=[0, .25,
                                                                            .5, .75, 1], labels=['Low', 'Medium low',
                                                                                                 'Medium high', 'High'])

#export for use in the next notebook
oews_with_scores.to_csv(outfile, index=False)

## Just checkout which codes didn't match real quick

In [ ]:
#codes that didn't match
list(oews_with_scores.loc[oews_with_scores['study_rating'].isna()]['occupation_code'].unique())

[Looking these missing codes up](https://www.bls.gov/oes/2023/may/oes_stru.htm) to make sure they're mostly "other" occupations:

- 119179 - Personal Service Managers, All Other
- 119199 - Managers, All Other
- 131199 - Business Operations Specialists, All Other
- 132099 - Financial Specialists, All Other
- 151299 - Computer Occupations, All Other
- 172199 - Engineers, All Other
- 173029 - Engineering Technologists and Technicians, Except Drafters, All Other
- 191029 - Biological Scientists, All Other
- 192099 - Physical Scientists, All Other
- 193039 - Psychologists, All Other
- 193099 - Social Scientists and Related Workers, All Other
- 194099 - Life, Physical, and Social Science Technicians, All Other
- 252059 - Special Education Teachers, All Other
- 291129 - Therapists, All Other
- 291229 - Physicians, All Other
- 291299 - Healthcare Diagnosing or Treating Practitioners, All Other
- 292099 - Health Technologists and Technicians, All Other
- 299099 - Healthcare Practitioners and Technical Workers, All Other
- 319099 - Healthcare Support Workers, All Other
- 339099 - Protective Service Workers, All Other
- 499099 - Installation, Maintenance, and Repair Workers, All Other
- 518099 - Plant and System Operators, All Other
- 152099 - Mathematical Science Occupations, All Other

In [ ]:
oews_with_scores.columns

In [ ]:
#show the top 5 occupations in each metro area by study rating
min_cols = ['location_name','occupation_name_oews','value','study_rating']
oews_with_scores.sort_values(['location_code', 'study_rating'], ascending=[True, False]).groupby('location_code')[min_cols].head(5)